# 01 — Data Exploration

Explore the sample medical reports, understand text distributions, and validate the ingestion pipeline.

## Goals
- Load and inspect sample reports
- Analyse text length, structure, and quality
- Validate lab-value regex extraction
- Visualise entity type distribution


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from collections import Counter
from rich import print as rprint

from src.extraction.ner_extractor import extract_lab_values
from src.utils.helpers import clean_text

print('Imports OK ✓')

## 1. Load Sample Report

In [ ]:
report_path = Path('../data/sample_reports/sample_blood_test.txt')
raw_text = report_path.read_text()
text = clean_text(raw_text)

print(f'File: {report_path.name}')
print(f'Characters: {len(text):,}')
print(f'Lines: {text.count(chr(10)):,}')
print(f'Words: {len(text.split()):,}')
print()
print('--- First 500 chars ---')
print(text[:500])

## 2. Lab Value Extraction

In [ ]:
lab_values = extract_lab_values(text)

df = pd.DataFrame(lab_values)
print(f'Extracted {len(df)} lab values\n')
if not df.empty:
    display(df[['test_name', 'value', 'unit']].head(20))

## 3. Value Distribution Visualisation

In [ ]:
if not df.empty:
    import numpy as np
    df['numeric'] = pd.to_numeric(df['value'], errors='coerce')
    df_valid = df.dropna(subset=['numeric'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram of numeric values
    axes[0].hist(df_valid['numeric'], bins=20, color='steelblue', edgecolor='white')
    axes[0].set_title('Distribution of Lab Values')
    axes[0].set_xlabel('Value')
    axes[0].set_ylabel('Count')

    # Top tests by value (bar chart)
    top_df = df_valid.nlargest(10, 'numeric')
    axes[1].barh(top_df['test_name'], top_df['numeric'], color='coral')
    axes[1].set_title('Top 10 Tests by Numeric Value')
    axes[1].set_xlabel('Value')

    plt.tight_layout()
    plt.savefig('../outputs/data_exploration.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/data_exploration.png')

## 4. Text Section Analysis

In [ ]:
sections = []
current = None
for line in text.splitlines():
    line = line.strip()
    if line.startswith('===') or (line.isupper() and len(line) > 5):
        current = line.strip('= ')
    elif current and line and not line.startswith('-'):
        sections.append({'section': current, 'line': line})

sec_df = pd.DataFrame(sections)
if not sec_df.empty:
    sec_counts = sec_df['section'].value_counts()
    print('Lines per section:')
    print(sec_counts.to_string())

## 5. Keyword Frequency

In [ ]:
medical_keywords = [
    'normal', 'low', 'high', 'critical', 'mg', 'g/dL', 'mIU', '%',
    'glucose', 'hemoglobin', 'cholesterol', 'creatinine', 'vitamin'
]

text_lower = text.lower()
freqs = {kw: text_lower.count(kw.lower()) for kw in medical_keywords}
freqs = dict(sorted(freqs.items(), key=lambda x: -x[1]))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(freqs.keys(), freqs.values(), color='teal', alpha=0.8)
ax.set_title('Keyword Frequency in Report')
ax.set_xlabel('Keyword')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../outputs/keyword_frequency.png', dpi=150)
plt.show()